# Step 1 — Dynamic Schema-Driven IFRS Requirement Mapping

This notebook maps all discovered IFRS S1/S2 requirements to all discovered payload sections **without hardcoding requirement IDs or payload fields**.

## How it works

1. Dynamically discovers the bank, payload files and requirement files.
2. Flattens each payload and converts repeated array indexes into generalized schema paths such as `records[].field_name`.
3. Builds a compact schema catalogue with types, sample values, years and record context.
4. Sends requirement batches and the schema catalogue to the configured fast LLM.
5. The LLM can select only schema IDs from the catalogue; it cannot invent paths.
6. Resolves selected schema IDs back to exact payload paths and groups fields by their original record.
7. Validates every resolved path and caches each requirement mapping.

No requirement-specific contracts, field-name lists or bank-specific filenames are used.

In [ ]:
# ============================================================
# CELL 1 — Imports, configuration and dynamic discovery
# ============================================================

import os
import re
import json
import math
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(items, **kwargs):
        return items


def locate_notebook_dir(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if candidate.name.lower() == "notebooks":
            return candidate
        nested = candidate / "notebooks"
        if nested.exists() and nested.is_dir():
            return nested.resolve()
    return start


CURRENT_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = locate_notebook_dir(CURRENT_DIR)
GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

PAYLOAD_DIR = Path(
    os.getenv("PAYLOAD_DIR", str(GEN_DATA_DIR / "payloads_risk"))
).resolve()

REQUIREMENTS_DIR = Path(
    os.getenv(
        "IFRS_REQUIREMENTS_DIR",
        str(
            GEN_DATA_DIR
            / "IFRS"
            / "ifrs_requirements_kb_outputs_final"
            / "section_by_section_requirements"
            / "json"
        ),
    )
).resolve()

GENERATION_OUTPUT_DIR = Path(
    os.getenv(
        "GENERATION_OUTPUT_DIR",
        str(GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report"),
    )
).resolve()

MAPPING_OUTPUT_DIR = Path(
    os.getenv(
        "MAPPING_OUTPUT_DIR",
        str(GENERATION_OUTPUT_DIR / "00_dynamic_requirement_mapping"),
    )
).resolve()

BANK_CODE = os.getenv("BANK_CODE", "").strip() or None
SECTIONS_TO_RUN = {
    value.strip()
    for value in os.getenv("MAPPING_SECTIONS", "").split(",")
    if value.strip()
}

USE_LLM_MAPPING = os.getenv("USE_LLM_MAPPING", "true").lower() in {
    "1", "true", "yes", "y"
}
STRICT_LLM_MAPPING = os.getenv("STRICT_LLM_MAPPING", "true").lower() in {
    "1", "true", "yes", "y"
}

LLM_BATCH_SIZE = int(os.getenv("MAPPING_LLM_BATCH_SIZE", "20"))
MAX_SCHEMA_SAMPLE_VALUES = int(os.getenv("MAX_SCHEMA_SAMPLE_VALUES", "4"))
MAX_SCHEMA_CONTEXT_VALUES = int(os.getenv("MAX_SCHEMA_CONTEXT_VALUES", "5"))
MAX_CONCRETE_PATHS_PER_SCHEMA = int(
    os.getenv("MAX_CONCRETE_PATHS_PER_SCHEMA", "100")
)
MAPPING_PROMPT_VERSION = "schema_mapper_v1_2026_07_28"

SECTION_ALIASES = {
    "general_requirements": "general_requirements",
    "general_requirement": "general_requirements",
    "governance": "governance",
    "strategy": "strategy",
    "risk_management": "risk_management",
    "risk": "risk_management",
    "metrics_targets": "metrics_and_targets",
    "metrics_and_targets": "metrics_and_targets",
    "metrics_target": "metrics_and_targets",
}


def normalize_section_key(value: str) -> str:
    value = re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")
    return SECTION_ALIASES.get(value, value)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def infer_payload_identity(path: Path) -> Tuple[Optional[str], Optional[str]]:
    stem = re.sub(r"\s*\(\d+\)$", "", path.stem).strip()
    section = None
    bank = None

    for alias in sorted(SECTION_ALIASES, key=len, reverse=True):
        suffix = "_" + alias
        if stem.lower().endswith(suffix):
            section = normalize_section_key(alias)
            prefix = stem[: -len(suffix)]
            bank = re.sub(r"^payload_", "", prefix, flags=re.IGNORECASE)
            break

    try:
        payload = load_json(path)
        metadata = payload.get("metadata", {}) if isinstance(payload, dict) else {}
        bank_object = payload.get("bank", {}) if isinstance(payload, dict) else {}
        bank = metadata.get("bank_id") or bank_object.get("bank_id") or bank
    except Exception:
        pass

    return (str(bank) if bank else None, section)


def discover_input_pairs() -> Tuple[str, Dict[str, Dict[str, Path]]]:
    if not PAYLOAD_DIR.exists():
        raise FileNotFoundError(f"Payload directory not found: {PAYLOAD_DIR}")
    if not REQUIREMENTS_DIR.exists():
        raise FileNotFoundError(
            f"Requirements directory not found: {REQUIREMENTS_DIR}"
        )

    requirement_files = {}
    for path in sorted(REQUIREMENTS_DIR.glob("*.json")):
        try:
            document = load_json(path)
        except Exception:
            continue

        if not isinstance(document, dict) or "standards" not in document:
            continue

        section = normalize_section_key(
            document.get("section_key")
            or path.stem.replace("_requirements", "")
        )
        if section in SECTION_ALIASES.values():
            requirement_files[section] = path

    payload_candidates = []
    for path in sorted(PAYLOAD_DIR.glob("payload_*.json")):
        bank, section = infer_payload_identity(path)
        if bank and section:
            payload_candidates.append((bank, section, path))

    available_banks = sorted({bank for bank, _, _ in payload_candidates})
    resolved_bank = BANK_CODE

    if resolved_bank is None:
        if len(available_banks) == 1:
            resolved_bank = available_banks[0]
        elif not available_banks:
            raise FileNotFoundError(
                f"No valid payload files found in {PAYLOAD_DIR}"
            )
        else:
            raise ValueError(
                "Multiple banks were found. Set BANK_CODE. "
                f"Available banks: {available_banks}"
            )

    payload_files = {
        section: path
        for bank, section, path in payload_candidates
        if bank == resolved_bank
    }

    sections = sorted(set(payload_files) & set(requirement_files))
    if SECTIONS_TO_RUN:
        requested = {normalize_section_key(x) for x in SECTIONS_TO_RUN}
        sections = [section for section in sections if section in requested]

    if not sections:
        raise ValueError(
            f"No matching payload/requirement pairs for bank {resolved_bank}."
        )

    pairs = {
        section: {
            "payload": payload_files[section],
            "requirements": requirement_files[section],
        }
        for section in sections
    }

    print("Bank:", resolved_bank)
    print("Payload directory:", PAYLOAD_DIR)
    print("Requirements directory:", REQUIREMENTS_DIR)
    print("Output directory:", MAPPING_OUTPUT_DIR)
    print("Mode:", "schema + LLM" if USE_LLM_MAPPING else "schema-only")

    for section, files in pairs.items():
        print(f"\n{section}")
        print("  payload:", files["payload"])
        print("  requirements:", files["requirements"])

    return resolved_bank, pairs


RESOLVED_BANK_CODE, RESOLVED_FILES = discover_input_pairs()

In [ ]:
# ============================================================
# CELL 2 — Dynamic flattening and generalized schema catalogue
# ============================================================

EMPTY_STRINGS = {
    "", "none", "null", "nan", "n/a", "na", "not available"
}


def is_empty(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip().lower() in EMPTY_STRINGS:
        return True
    if isinstance(value, (list, dict)) and not value:
        return True
    return False


def preview(value: Any, limit: int = 220) -> str:
    text = (
        json.dumps(value, ensure_ascii=False)
        if isinstance(value, (dict, list))
        else str(value)
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("…" if len(text) > limit else "")


def generalized_path(path: str) -> str:
    return re.sub(r"\[\d+\]", "[]", path)


def detect_context_fields(record: Dict[str, Any]) -> Dict[str, Any]:
    """
    Detect record identifiers and descriptive context from the schema itself.
    No section-specific key list is used.
    """
    context = {}
    for key, value in record.items():
        if isinstance(value, (dict, list)) or is_empty(value):
            continue

        normalized = str(key).lower()
        if (
            normalized == "reporting_year"
            or normalized.endswith("_id")
            or normalized.endswith("_year")
            or normalized.endswith("_date")
            or normalized.endswith("_name")
            or normalized.endswith("_type")
            or normalized.endswith("_category")
            or normalized.endswith("_status")
            or normalized.endswith("_scope")
            or normalized.endswith("_horizon")
        ):
            context[key] = value

    return context


def flatten_payload(
    value: Any,
    prefix: str = "",
    inherited_context: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    inherited_context = dict(inherited_context or {})
    rows = []

    if isinstance(value, dict):
        local_context = {
            **inherited_context,
            **detect_context_fields(value),
        }

        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)

            if isinstance(child, (dict, list)):
                rows.extend(
                    flatten_payload(child, path, local_context)
                )
            else:
                rows.append({
                    "path": path,
                    "schema_path": generalized_path(path),
                    "parent_path": prefix,
                    "schema_parent_path": generalized_path(prefix),
                    "root": path.split(".", 1)[0].split("[", 1)[0],
                    "leaf": str(key),
                    "value": child,
                    "value_type": type(child).__name__,
                    "context": local_context,
                })

    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(
                flatten_payload(
                    child,
                    f"{prefix}[{index}]",
                    inherited_context,
                )
            )

    return rows


def unique_preserving_order(values: List[Any]) -> List[Any]:
    seen = set()
    result = []

    for value in values:
        marker = json.dumps(value, sort_keys=True, ensure_ascii=False)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(value)

    return result


def build_schema_catalog(
    payload: Dict[str, Any],
) -> Tuple[
    List[Dict[str, Any]],
    Dict[str, Dict[str, Any]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    flat_rows = [
        row for row in flatten_payload(payload)
        if not is_empty(row["value"])
    ]

    grouped = defaultdict(list)
    path_index = {}

    for row in flat_rows:
        grouped[row["schema_path"]].append(row)
        path_index[row["path"]] = row["value"]

    schema_catalog = []
    schema_by_id = {}
    rows_by_schema_id = {}

    for index, schema_path in enumerate(sorted(grouped), start=1):
        rows = grouped[schema_path]
        schema_id = f"S{index:04d}"

        sample_values = unique_preserving_order([
            preview(row["value"], 160) for row in rows
        ])[:MAX_SCHEMA_SAMPLE_VALUES]

        context_summary = defaultdict(list)
        for row in rows:
            for key, value in row["context"].items():
                context_summary[key].append(value)

        context_summary = {
            key: unique_preserving_order(values)[
                :MAX_SCHEMA_CONTEXT_VALUES
            ]
            for key, values in context_summary.items()
        }

        schema_item = {
            "schema_id": schema_id,
            "schema_path": schema_path,
            "root": rows[0]["root"],
            "leaf": rows[0]["leaf"],
            "value_types": sorted({
                row["value_type"] for row in rows
            }),
            "record_count": len({
                row["parent_path"] for row in rows
            }),
            "value_count": len(rows),
            "sample_values": sample_values,
            "context_summary": context_summary,
        }

        schema_catalog.append(schema_item)
        schema_by_id[schema_id] = schema_item
        rows_by_schema_id[schema_id] = rows

    return schema_catalog, schema_by_id, rows_by_schema_id, path_index


def iter_requirements(document: Dict[str, Any]) -> List[Dict[str, Any]]:
    requirements = []
    for block in document.get("standards", {}).values():
        requirements.extend(block.get("requirements", []))
    return requirements


def compact_schema_for_prompt(
    schema_catalog: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    return [
        {
            "schema_id": row["schema_id"],
            "path": row["schema_path"],
            "types": row["value_types"],
            "samples": row["sample_values"],
            "context": row["context_summary"],
            "record_count": row["record_count"],
        }
        for row in schema_catalog
    ]

In [ ]:
# ============================================================
# CELL 3 — LLM schema selection and cache
# ============================================================

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv(
    "AZURE_OPENAI_FAST_DEPLOYMENT_URL"
)
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv(
    "AZURE_OPENAI_GPT52_DEPLOYMENT_URL"
)
MAPPING_LLM_URL = (
    AZURE_OPENAI_FAST_DEPLOYMENT_URL
    or AZURE_OPENAI_GPT52_DEPLOYMENT_URL
)

VALID_MAPPING_STATUSES = {
    "covered",
    "partially_covered",
    "not_available_in_payload",
    "conditional_not_triggered",
    "handled_by_report_design",
    "not_applicable_to_entity_scope",
}

VALID_RECORD_SCOPES = {
    "current_period",
    "comparative_periods",
    "all_records",
    "non_period_specific",
    "representative_records",
}


def parse_json_object(raw: str) -> Dict[str, Any]:
    raw = str(raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start < 0 or end <= start:
            raise
        return json.loads(raw[start:end + 1])


def standalone_azure_json(
    messages: List[Dict[str, str]],
    max_tokens: int = 12000,
    retries: int = 5,
) -> Dict[str, Any]:
    if not AZURE_OPENAI_API_KEY or not MAPPING_LLM_URL:
        raise ValueError(
            "LLM mapping is enabled but Azure/OpenAI configuration is missing. "
            "Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_FAST_DEPLOYMENT_URL, "
            "or set USE_LLM_MAPPING=false."
        )

    last_error = None

    for token_field in ("max_completion_tokens", "max_tokens"):
        request_body = {
            "messages": messages,
            token_field: max_tokens,
            "response_format": {"type": "json_object"},
        }

        for attempt in range(1, retries + 1):
            request = urllib.request.Request(
                MAPPING_LLM_URL,
                data=json.dumps(request_body).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": AZURE_OPENAI_API_KEY,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(
                    request,
                    timeout=240,
                ) as response:
                    result = json.loads(
                        response.read().decode("utf-8")
                    )

                content = result["choices"][0]["message"]["content"]
                return parse_json_object(content)

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"Mapping LLM HTTP {exc.code}: {body[:1200]}"
                )

                if exc.code == 400 and token_field == "max_completion_tokens":
                    break
                if exc.code == 429 or exc.code >= 500:
                    time.sleep(min(30, attempt * 4))
                    continue
                raise last_error

            except Exception as exc:
                last_error = exc
                if attempt < retries:
                    time.sleep(min(30, attempt * 4))
                    continue
                break

    raise RuntimeError(f"Mapping LLM failed: {last_error}")


def call_mapping_llm(
    messages: List[Dict[str, str]],
) -> Dict[str, Any]:
    existing_helper = globals().get("azure_chat_json")

    if callable(existing_helper):
        return existing_helper(
            messages=messages,
            model_tier="fast",
            temperature=0,
            max_tokens=12000,
            request_label="dynamic schema evidence mapper",
        )

    return standalone_azure_json(messages)


SCHEMA_MAPPER_SYSTEM_PROMPT = """
You map IFRS S1 and IFRS S2 requirements to a supplied payload schema.

The schema catalogue contains the only fields you may select. Select schema IDs,
not invented paths.

Rules:
1. Map the exact disclosure requirement, not merely related subject matter.
2. Do not infer a policy, process, control, responsibility, trade-off,
   methodology or explanation from a flag, outcome or metric.
3. Do not use a numeric match when the field's semantic meaning differs.
4. Use covered only when selected schema fields can directly support all
   material parts of the requirement.
5. Use partially_covered when direct fields support only part; specify exactly
   what is missing.
6. Use not_available_in_payload when no schema field directly supports it.
7. Use conditional_not_triggered only when the requirement itself is
   conditional and the trigger is absent.
8. Use handled_by_report_design only for report presentation, duplication,
   location, cross-reference or assembly controls.
9. Use not_applicable_to_entity_scope only when the schema demonstrates that
   the applicable business activity is outside entity scope.
10. Choose record_scope:
    - current_period: current reporting-year records only;
    - comparative_periods: current and prior-period records;
    - all_records: every matching record is required;
    - non_period_specific: fields without a reporting-year context;
    - representative_records: a small sample is sufficient to prove the
      process or structure.
11. Return JSON only.

Return:
{
  "mappings": [
    {
      "requirement_id": "...",
      "mapping_status": "covered|partially_covered|not_available_in_payload|conditional_not_triggered|handled_by_report_design|not_applicable_to_entity_scope",
      "evidence": [
        {
          "schema_id": "S0001",
          "record_scope": "current_period",
          "support_reason": "..."
        }
      ],
      "missing_information": ["..."],
      "rationale": "...",
      "confidence": 0.0
    }
  ]
}
""".strip()


def requirement_for_prompt(
    requirement: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "requirement_id": requirement["requirement_id"],
        "standard": requirement.get("standard"),
        "paragraph_id": requirement.get("paragraph_id"),
        "clause_path": requirement.get("clause_path"),
        "requirement_text": requirement.get("requirement_text"),
        "official_section_heading": requirement.get(
            "official_section_heading"
        ),
        "evidence_tags": requirement.get("evidence_tags") or [],
        "mandatory": requirement.get("mandatory"),
    }


def cache_key(
    requirement: Dict[str, Any],
    schema_catalog: List[Dict[str, Any]],
) -> str:
    content = {
        "prompt_version": MAPPING_PROMPT_VERSION,
        "requirement": requirement_for_prompt(requirement),
        "schema": compact_schema_for_prompt(schema_catalog),
    }
    return hashlib.sha256(
        json.dumps(
            content,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()


def cache_path(
    cache_dir: Path,
    requirement: Dict[str, Any],
    schema_catalog: List[Dict[str, Any]],
) -> Path:
    key = cache_key(requirement, schema_catalog)
    return cache_dir / (
        f"{requirement['requirement_id']}__{key}.json"
    )


def load_cached_result(
    cache_dir: Path,
    requirement: Dict[str, Any],
    schema_catalog: List[Dict[str, Any]],
) -> Optional[Dict[str, Any]]:
    path = cache_path(cache_dir, requirement, schema_catalog)
    return load_json(path) if path.exists() else None


def save_cached_result(
    cache_dir: Path,
    requirement: Dict[str, Any],
    schema_catalog: List[Dict[str, Any]],
    result: Dict[str, Any],
) -> None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(cache_dir, requirement, schema_catalog)
    path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def validate_llm_mapping(
    requirement: Dict[str, Any],
    raw: Dict[str, Any],
    schema_by_id: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    requirement_id = requirement["requirement_id"]

    if raw.get("requirement_id") != requirement_id:
        raise ValueError(
            f"Requirement mismatch: expected {requirement_id}, "
            f"received {raw.get('requirement_id')}"
        )

    status = raw.get("mapping_status")
    if status not in VALID_MAPPING_STATUSES:
        raise ValueError(
            f"Invalid status for {requirement_id}: {status}"
        )

    evidence = raw.get("evidence") or []
    validated_evidence = []

    for item in evidence:
        schema_id = item.get("schema_id")
        record_scope = item.get("record_scope")

        if schema_id not in schema_by_id:
            raise ValueError(
                f"{requirement_id} selected unknown schema ID {schema_id}"
            )
        if record_scope not in VALID_RECORD_SCOPES:
            raise ValueError(
                f"{requirement_id} returned invalid record scope "
                f"{record_scope}"
            )

        validated_evidence.append({
            "schema_id": schema_id,
            "schema_path": schema_by_id[schema_id]["schema_path"],
            "record_scope": record_scope,
            "support_reason": str(
                item.get("support_reason", "")
            ).strip(),
        })

    evidence_statuses = {"covered", "partially_covered"}
    non_evidence_statuses = VALID_MAPPING_STATUSES - evidence_statuses

    if status in evidence_statuses and not validated_evidence:
        raise ValueError(
            f"{requirement_id} returned {status} without evidence."
        )
    if status in non_evidence_statuses and validated_evidence:
        raise ValueError(
            f"{requirement_id} returned {status} with evidence."
        )

    return {
        "requirement_id": requirement_id,
        "mapping_status": status,
        "schema_evidence": validated_evidence,
        "missing_information": raw.get("missing_information") or [],
        "mapping_rationale": str(raw.get("rationale", "")).strip(),
        "mapping_confidence": float(raw.get("confidence", 0.0)),
        "mapping_method": "dynamic_schema_plus_llm",
        "prompt_version": MAPPING_PROMPT_VERSION,
    }


def map_requirement_batch(
    requirements: List[Dict[str, Any]],
    schema_catalog: List[Dict[str, Any]],
    schema_by_id: Dict[str, Dict[str, Any]],
) -> List[Dict[str, Any]]:
    payload = {
        "requirements": [
            requirement_for_prompt(requirement)
            for requirement in requirements
        ],
        "payload_schema": compact_schema_for_prompt(schema_catalog),
    }

    response = call_mapping_llm([
        {"role": "system", "content": SCHEMA_MAPPER_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(payload, ensure_ascii=False),
        },
    ])

    returned = {
        item.get("requirement_id"): item
        for item in response.get("mappings", [])
        if isinstance(item, dict)
    }

    results = []
    for requirement in requirements:
        requirement_id = requirement["requirement_id"]
        if requirement_id not in returned:
            raise ValueError(
                f"LLM omitted requirement {requirement_id}"
            )
        results.append(
            validate_llm_mapping(
                requirement,
                returned[requirement_id],
                schema_by_id,
            )
        )

    return results

In [ ]:
# ============================================================
# CELL 4 — Resolve schema IDs to exact record-aware paths
# ============================================================

def reporting_year_of(payload: Dict[str, Any]) -> Optional[int]:
    value = payload.get("metadata", {}).get("reporting_year")
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def row_reporting_year(row: Dict[str, Any]) -> Optional[int]:
    value = row.get("context", {}).get("reporting_year")
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def select_concrete_rows(
    rows: List[Dict[str, Any]],
    record_scope: str,
    reporting_year: Optional[int],
) -> List[Dict[str, Any]]:
    if record_scope == "current_period":
        selected = [
            row for row in rows
            if row_reporting_year(row) == reporting_year
        ]
        if selected:
            return selected

        # Static metadata or policy fields have no period context.
        return [
            row for row in rows
            if row_reporting_year(row) is None
        ]

    if record_scope == "comparative_periods":
        selected = [
            row for row in rows
            if (
                row_reporting_year(row) is not None
                and reporting_year is not None
                and row_reporting_year(row) <= reporting_year
            )
        ]
        return selected or rows

    if record_scope == "non_period_specific":
        selected = [
            row for row in rows
            if row_reporting_year(row) is None
        ]
        return selected

    if record_scope == "representative_records":
        current = [
            row for row in rows
            if row_reporting_year(row) == reporting_year
        ]
        source = current or rows

        selected = []
        seen_parents = set()
        for row in source:
            if row["parent_path"] in seen_parents:
                continue
            seen_parents.add(row["parent_path"])
            selected.append(row)
            if len(selected) >= 5:
                break
        return selected

    return rows


def resolve_mapping_evidence(
    mapping: Dict[str, Any],
    rows_by_schema_id: Dict[str, List[Dict[str, Any]]],
    payload: Dict[str, Any],
) -> Dict[str, Any]:
    reporting_year = reporting_year_of(payload)
    concrete_rows = []

    for evidence in mapping["schema_evidence"]:
        schema_id = evidence["schema_id"]
        selected = select_concrete_rows(
            rows_by_schema_id[schema_id],
            evidence["record_scope"],
            reporting_year,
        )

        if not selected:
            raise ValueError(
                f"{mapping['requirement_id']} selected {schema_id} "
                f"with scope {evidence['record_scope']}, but no records resolved."
            )

        for row in selected[:MAX_CONCRETE_PATHS_PER_SCHEMA]:
            concrete_rows.append({
                "schema_id": schema_id,
                "schema_path": evidence["schema_path"],
                "record_scope": evidence["record_scope"],
                "support_reason": evidence["support_reason"],
                "path": row["path"],
                "parent_path": row["parent_path"],
                "value": row["value"],
                "value_type": row["value_type"],
                "context": row["context"],
            })

    # Remove exact duplicates without collapsing different records.
    deduplicated = []
    seen = set()

    for row in concrete_rows:
        key = (row["schema_id"], row["path"])
        if key in seen:
            continue
        seen.add(key)
        deduplicated.append(row)

    evidence_groups = defaultdict(list)
    for row in deduplicated:
        evidence_groups[row["parent_path"]].append(row)

    grouped_output = [
        {
            "record_path": record_path,
            "record_context": next(
                (
                    field["context"]
                    for field in fields
                    if field["context"]
                ),
                {},
            ),
            "fields": fields,
        }
        for record_path, fields in sorted(evidence_groups.items())
    ]

    return {
        **mapping,
        "selected_evidence_paths": [
            row["path"] for row in deduplicated
        ],
        "selected_evidence": deduplicated,
        "evidence_groups": grouped_output,
    }

In [ ]:
# ============================================================
# CELL 5 — Dynamic all-section mapping orchestrator
# ============================================================

def schema_only_result(
    requirement: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "requirement_id": requirement["requirement_id"],
        "mapping_status": "needs_semantic_validation",
        "schema_evidence": [],
        "missing_information": [],
        "mapping_rationale": (
            "Schema catalogue built successfully, but LLM schema selection "
            "was disabled. No coverage status has been asserted."
        ),
        "mapping_confidence": 0.0,
        "mapping_method": "dynamic_schema_only",
        "prompt_version": MAPPING_PROMPT_VERSION,
        "selected_evidence_paths": [],
        "selected_evidence": [],
        "evidence_groups": [],
    }


def map_section(
    section: str,
    payload: Dict[str, Any],
    requirements_document: Dict[str, Any],
) -> Tuple[
    List[Dict[str, Any]],
    List[Dict[str, Any]],
    Dict[str, Any],
]:
    requirements = iter_requirements(requirements_document)

    (
        schema_catalog,
        schema_by_id,
        rows_by_schema_id,
        path_index,
    ) = build_schema_catalog(payload)

    cache_dir = MAPPING_OUTPUT_DIR / "cache" / section
    mapping_by_requirement = {}
    uncached_requirements = []

    for requirement in requirements:
        if not USE_LLM_MAPPING:
            mapping_by_requirement[
                requirement["requirement_id"]
            ] = schema_only_result(requirement)
            continue

        cached = load_cached_result(
            cache_dir,
            requirement,
            schema_catalog,
        )

        if cached is not None:
            mapping_by_requirement[
                requirement["requirement_id"]
            ] = resolve_mapping_evidence(
                cached,
                rows_by_schema_id,
                payload,
            )
        else:
            uncached_requirements.append(requirement)

    if USE_LLM_MAPPING:
        for start in tqdm(
            range(0, len(uncached_requirements), LLM_BATCH_SIZE),
            desc=f"Schema mapping — {section}",
        ):
            batch = uncached_requirements[
                start:start + LLM_BATCH_SIZE
            ]

            try:
                batch_results = map_requirement_batch(
                    batch,
                    schema_catalog,
                    schema_by_id,
                )
            except Exception:
                if STRICT_LLM_MAPPING:
                    raise
                batch_results = [
                    schema_only_result(requirement)
                    for requirement in batch
                ]

            for requirement, mapping in zip(
                batch,
                batch_results,
            ):
                if mapping["mapping_method"] == "dynamic_schema_plus_llm":
                    save_cached_result(
                        cache_dir,
                        requirement,
                        schema_catalog,
                        mapping,
                    )
                    mapping = resolve_mapping_evidence(
                        mapping,
                        rows_by_schema_id,
                        payload,
                    )

                mapping_by_requirement[
                    requirement["requirement_id"]
                ] = mapping

    rows = []
    for requirement in requirements:
        mapping = mapping_by_requirement[
            requirement["requirement_id"]
        ]

        rows.append({
            "section_key": section,
            "requirement_id": requirement["requirement_id"],
            "standard": requirement.get("standard"),
            "paragraph_id": requirement.get("paragraph_id"),
            "clause_path": requirement.get("clause_path"),
            "official_section_heading": requirement.get(
                "official_section_heading"
            ),
            "requirement_text": requirement.get("requirement_text"),
            "evidence_tags": requirement.get("evidence_tags") or [],
            "mandatory": requirement.get("mandatory"),
            **mapping,
        })

    return rows, schema_catalog, path_index


payloads = {
    section: load_json(files["payload"])
    for section, files in RESOLVED_FILES.items()
}
requirements_documents = {
    section: load_json(files["requirements"])
    for section, files in RESOLVED_FILES.items()
}

mappings_by_section = {}
schema_catalogs_by_section = {}
path_indexes_by_section = {}

for section in RESOLVED_FILES:
    (
        mappings_by_section[section],
        schema_catalogs_by_section[section],
        path_indexes_by_section[section],
    ) = map_section(
        section,
        payloads[section],
        requirements_documents[section],
    )

print("Sections mapped:", list(mappings_by_section))
print(
    "Total requirements:",
    sum(len(rows) for rows in mappings_by_section.values()),
)

In [ ]:
# ============================================================
# CELL 6 — Validation and outputs
# ============================================================

def validate_section(
    section: str,
    mappings: List[Dict[str, Any]],
    requirements_document: Dict[str, Any],
    path_index: Dict[str, Any],
) -> Dict[str, Any]:
    source_requirements = iter_requirements(
        requirements_document
    )
    source_ids = [
        row["requirement_id"] for row in source_requirements
    ]
    mapped_ids = [
        row["requirement_id"] for row in mappings
    ]

    invalid_paths = []
    empty_paths = []
    duplicate_paths = []
    invalid_statuses = []

    for row in mappings:
        paths = row["selected_evidence_paths"]

        if len(paths) != len(set(paths)):
            duplicate_paths.append(row["requirement_id"])

        if (
            row["mapping_status"] not in VALID_MAPPING_STATUSES
            and row["mapping_status"] != "needs_semantic_validation"
        ):
            invalid_statuses.append(row["requirement_id"])

        for path in paths:
            if path not in path_index:
                invalid_paths.append({
                    "requirement_id": row["requirement_id"],
                    "path": path,
                })
            elif is_empty(path_index[path]):
                empty_paths.append({
                    "requirement_id": row["requirement_id"],
                    "path": path,
                })

    tests = {
        "requirement_count_matches": (
            len(mappings)
            == len(source_requirements)
            == requirements_document.get("row_count")
        ),
        "all_requirement_ids_mapped": (
            set(mapped_ids) == set(source_ids)
        ),
        "requirement_ids_unique": (
            len(mapped_ids) == len(set(mapped_ids))
        ),
        "all_selected_paths_resolve": not invalid_paths,
        "all_selected_values_non_empty": not empty_paths,
        "no_duplicate_selected_paths": not duplicate_paths,
        "all_statuses_valid": not invalid_statuses,
        "covered_and_partial_have_evidence": all(
            row["selected_evidence_paths"]
            for row in mappings
            if row["mapping_status"] in {
                "covered", "partially_covered"
            }
        ),
        "non_evidence_statuses_have_no_paths": all(
            not row["selected_evidence_paths"]
            for row in mappings
            if row["mapping_status"] in {
                "not_available_in_payload",
                "conditional_not_triggered",
                "handled_by_report_design",
                "not_applicable_to_entity_scope",
                "needs_semantic_validation",
            }
        ),
    }

    return {
        "section": section,
        "result": "PASS" if all(tests.values()) else "FAIL",
        "tests": tests,
        "invalid_paths": invalid_paths,
        "empty_paths": empty_paths,
        "duplicate_paths": duplicate_paths,
        "invalid_statuses": invalid_statuses,
    }


test_reports = {
    section: validate_section(
        section,
        mappings_by_section[section],
        requirements_documents[section],
        path_indexes_by_section[section],
    )
    for section in mappings_by_section
}

summary = {
    "bank_code": RESOLVED_BANK_CODE,
    "mapping_method": (
        "dynamic_schema_plus_llm"
        if USE_LLM_MAPPING
        else "dynamic_schema_only"
    ),
    "prompt_version": MAPPING_PROMPT_VERSION,
    "total_requirements": sum(
        len(rows) for rows in mappings_by_section.values()
    ),
    "schema_field_counts": {
        section: len(schema)
        for section, schema in schema_catalogs_by_section.items()
    },
    "status_totals": dict(Counter(
        row["mapping_status"]
        for rows in mappings_by_section.values()
        for row in rows
    )),
    "sections": {},
}

for section, rows in mappings_by_section.items():
    confidence_values = [
        row["mapping_confidence"]
        for row in rows
        if row["mapping_method"] == "dynamic_schema_plus_llm"
    ]

    summary["sections"][section] = {
        "requirement_count": len(rows),
        "schema_field_count": len(
            schema_catalogs_by_section[section]
        ),
        "status_counts": dict(Counter(
            row["mapping_status"] for row in rows
        )),
        "average_confidence": round(
            float(np.mean(confidence_values))
            if confidence_values else 0.0,
            4,
        ),
    }

combined_test_report = {
    "result": (
        "PASS"
        if all(
            report["result"] == "PASS"
            for report in test_reports.values()
        )
        else "FAIL"
    ),
    "sections": test_reports,
}

MAPPING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for section, rows in mappings_by_section.items():
    (MAPPING_OUTPUT_DIR / f"{section}_dynamic_mapping.json").write_text(
        json.dumps(rows, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (MAPPING_OUTPUT_DIR / f"{section}_schema_catalog.json").write_text(
        json.dumps(
            schema_catalogs_by_section[section],
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    (MAPPING_OUTPUT_DIR / f"{section}_mapping_test.json").write_text(
        json.dumps(
            test_reports[section],
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    pd.DataFrame([
        {
            "requirement_id": row["requirement_id"],
            "standard": row["standard"],
            "paragraph_id": row["paragraph_id"],
            "mapping_status": row["mapping_status"],
            "confidence": row["mapping_confidence"],
            "selected_schema_paths": " | ".join(
                evidence["schema_path"]
                for evidence in row["schema_evidence"]
            ),
            "selected_exact_paths": " | ".join(
                row["selected_evidence_paths"]
            ),
            "missing_information": " | ".join(
                row["missing_information"]
            ),
        }
        for row in rows
    ]).to_csv(
        MAPPING_OUTPUT_DIR / f"{section}_dynamic_mapping.csv",
        index=False,
        encoding="utf-8-sig",
    )

(MAPPING_OUTPUT_DIR / "all_sections_dynamic_mapping.json").write_text(
    json.dumps(
        mappings_by_section,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
(MAPPING_OUTPUT_DIR / "all_sections_mapping_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(MAPPING_OUTPUT_DIR / "all_sections_mapping_test.json").write_text(
    json.dumps(
        combined_test_report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("\nTEST RESULT:", combined_test_report["result"])
print("Outputs:", MAPPING_OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 7 — Interactive inspection
# ============================================================

def inspect_mapping(
    requirement_id: str,
    section: Optional[str] = None,
) -> None:
    section_keys = (
        [normalize_section_key(section)]
        if section
        else list(mappings_by_section)
    )

    for section_key in section_keys:
        for row in mappings_by_section.get(section_key, []):
            if row["requirement_id"] != requirement_id:
                continue

            print("Section:", section_key)
            print("Requirement:", requirement_id)
            print("Status:", row["mapping_status"])
            print("Confidence:", row["mapping_confidence"])
            print("\nRequirement text:")
            print(row["requirement_text"])

            print("\nSelected schema:")
            for evidence in row["schema_evidence"]:
                print(
                    f"- {evidence['schema_id']} "
                    f"{evidence['schema_path']} "
                    f"[{evidence['record_scope']}]"
                )

            print("\nResolved evidence groups:")
            for group in row["evidence_groups"]:
                print("\nRecord:", group["record_path"])
                print("Context:", group["record_context"])
                for field in group["fields"]:
                    print(
                        f"  - {field['path']} = "
                        f"{preview(field['value'], 180)}"
                    )

            print("\nMissing information:")
            for item in row["missing_information"]:
                print("-", item)

            print("\nRationale:")
            print(row["mapping_rationale"])
            return

    raise KeyError(f"Requirement not found: {requirement_id}")


# Example:
# inspect_mapping("<requirement_id>", section="<section_key>")